# 03 - Extract gridMET 4 km observed baseline

Daily `pr`, `tmmx`, `tmmn`, `vs` 1979-present from the Climatology Lab THREDDS server.
The aggregated OPeNDAP endpoints (`agg_met_<var>_1979_CurrentYear_CONUS.nc`) let the
server do the bbox subsetting - one consolidated local NetCDF per variable lands in
`data/raw/gridmet/`. Runs with the stock arcgispro-py3 (no extra packages).

In [ ]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("03_extract_gridmet")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

In [ ]:
G = cfg["sources"]["gridmet"]
gm_dir = RAW / "gridmet"
gm_dir.mkdir(exist_ok=True)

for var in G["variables"]:
    url = f"{G['dodsc_base']}/{G['agg_pattern'].format(var=var)}"
    out = gm_dir / f"gridmet_{var}_tahoe.nc"
    if out.exists() and not cfg["run"]["overwrite_downloads"]:
        log.info(f"{out.name} present - skipped")
        continue
    try:
        ds = xr.open_dataset(url)   # lazy OPeNDAP
        lat_asc = bool(ds.lat[0] < ds.lat[-1])
        sub = ds.sel(lon=slice(BBOX["lon_min"], BBOX["lon_max"]),
                     lat=slice(BBOX["lat_min"], BBOX["lat_max"]) if lat_asc
                     else slice(BBOX["lat_max"], BBOX["lat_min"])).load()
        sub.to_netcdf(out, encoding={v: {"zlib": True, "complevel": 4}
                                     for v in sub.data_vars})
        append_manifest({"file": str(out.relative_to(ROOT)), "source_url": url,
                         "size_bytes": out.stat().st_size, "sha256": sha256_file(out),
                         "retrieved_date": str(pd.Timestamp.today().date()),
                         "notebook": "03_extract_gridmet"})
        dv = list(sub.data_vars)[0]
        log.info(f"{out.name}: {out.stat().st_size/1e6:.1f} MB, var={dv}, "
                 f"days={sub.sizes.get('day', '?')}, "
                 f"grid={sub.sizes.get('lat')}x{sub.sizes.get('lon')}")
    except Exception as e:
        log.warning(f"FAILED {var}: {e}")

log.info("gridMET extract complete")